In [ ]:
import os
import json
import time
from openai import OpenAI
from tqdm import tqdm

# --- 설정 ---
INPUT_PROMPTS_FILE = 'llm_prompt/동원참치액 순 500g_prompts_for_llm.jsonl'  # 업로드 파일 경로 권장
OUTPUT_RESULTS_FILE = 'llm_results/동원참치액 순 500g_llm_results.jsonl'
OPENAI_MODEL = "gpt-4o-mini"
SLEEP_TIME_BETWEEN_REQUESTS = 1
# --- 디버그 옵션 ---
DEBUG_PROMPT_PRINT = True     # 콘솔에 프롬프트 출력
DEBUG_PROMPT_SAVE  = True    # 파일로 저장 (길 때 유용)

# --- 테스트용 개수 제한 ---
# LIMIT_PROMPTS = 10  # 필요 시 테스트 제한

# --- 유틸 ---
def ensure_parent_dir(path: str):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)

# --- OpenAI API 클라이언트 ---
def get_openai_client():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY 환경 변수가 설정되지 않았습니다.")
    return OpenAI(api_key=api_key)

# --- API 호출 (JSON 모드 → 실패 시 fallback) ---
def get_llm_prediction(client, system_prompt, user_prompt):
    try:
        # 1차 시도: JSON 모드
        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.0,
            seed=0,  # 재현성 더 강화하고 싶으면 활성화
        )
        return resp.choices[0].message.content
    except Exception as e:
        # response_format 미지원 등일 때 재시도
        if "response_format" in str(e) or "json_object" in str(e):
            try:
                resp = client.chat.completions.create(
                    model=OPENAI_MODEL,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=0.0,
                )
                return resp.choices[0].message.content
            except Exception as e2:
                print(f"JSON 모드 fallback도 실패: {e2}")
                return None
        else:
            print(f"API 호출 중 오류: {e}")
            return None

print("설정 및 함수 정의 완료.")

# === 메인 실행 ===
try:
    client = get_openai_client()

    with open(INPUT_PROMPTS_FILE, 'r', encoding='utf-8') as f:
        prompts = [json.loads(line) for line in f]
    print(f"총 {len(prompts)}개의 개선된 프롬프트를 불러왔습니다.")

    # === 테스트용  개수 제한 ===
    # if 'LIMIT_PROMPTS' in locals() and LIMIT_PROMPTS is not None:
    #     prompts = prompts[:LIMIT_PROMPTS]
    #     print(f"[테스트] 프롬프트 {len(prompts)}개만 실행합니다.")
    # === 테스트 종료 ===

    processed_keys = set()
    try:
        with open(OUTPUT_RESULTS_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line)
                rec = data.get("prediction", data)  # wrapper가 있으면 안쪽을, 없으면 그대로
                p_name = rec.get("product_name")
                p_key  = rec.get("persona_key")
                if p_name and p_key:
                    processed_keys.add((p_name, p_key))
        print(f"이미 처리된 결과 {len(processed_keys)}개 존재. 이어서 실행합니다.")
    except FileNotFoundError:
        print("결과 파일 신규 생성 예정.")

    ensure_parent_dir(OUTPUT_RESULTS_FILE)
    with open(OUTPUT_RESULTS_FILE, 'a', encoding='utf-8') as f_out:
        for prompt_data in tqdm(prompts, desc="LLM 예측 실행 중"):
            product_name = prompt_data.get('product_name')
            persona_key  = prompt_data.get('persona_key')
            if (product_name, persona_key) in processed_keys:
                continue

            system_prompt = prompt_data.get('system_prompt')
            user_prompt = prompt_data.get('user_prompt')

            # ===== 여기부터 프롬프트 디버그 출력/저장 (추가) =====
            if DEBUG_PROMPT_PRINT:
                print("\n===== SYSTEM =====\n")
                print(system_prompt)

                print("\n===== USER =====\n")
                print(user_prompt)                 # 사람이 읽기 좋은 원문
                # print(repr(user_prompt))         # \n, \t 등 이스케이프까지 보고 싶으면 이 줄 사용

            if DEBUG_PROMPT_SAVE:
                dbg_dir = "debug_prompts"
                ensure_parent_dir(os.path.join(dbg_dir, "_"))  # 폴더 생성
                # 제품명/페르소나가 있다면 파일명에 같이 남기면 구분 쉬움
                product_name = prompt_data.get('product_name', 'unknown_product')
                persona_key  = prompt_data.get('persona_key', 'unknown_persona')
                base = f"{product_name}__{persona_key}"

                with open(os.path.join(dbg_dir, f"{base}.system.txt"), "w", encoding="utf-8") as fs:
                    fs.write(system_prompt or "")
                with open(os.path.join(dbg_dir, f"{base}.user.txt"), "w", encoding="utf-8") as fu:
                    fu.write(user_prompt or "")
            # ===== 디버그 블록 끝 =====

            llm_response_str = get_llm_prediction(client, system_prompt, user_prompt)
            if llm_response_str:
                try:
                    # JSON 모드면 바로 파싱됨. 혹시 모를 텍스트 혼입도 방어
                    if '---' in llm_response_str:
                        _, json_part = llm_response_str.split('---', 1)
                        json_text = json_part[json_part.find('{'):json_part.rfind('}')+1]
                        llm_json_result = json.loads(json_text)
                    else:
                        llm_json_result = json.loads(llm_response_str)

                    final_result = {
                        "prediction": llm_json_result
                    }
                    f_out.write(json.dumps(final_result, ensure_ascii=False) + '\n')

                except (json.JSONDecodeError, ValueError) as e:
                    print(f"\n파싱 실패: {e}\n응답: {llm_response_str}\n")

            time.sleep(SLEEP_TIME_BETWEEN_REQUESTS)

    print("="*40)
    print(f"🎉 완료: '{OUTPUT_RESULTS_FILE}'에 저장되었습니다.")

except Exception as e:
    print(f"실행 중 오류: {e}")


설정 및 함수 정의 완료.
총 100개의 개선된 프롬프트를 불러왔습니다.
결과 파일 신규 생성 예정.


LLM 예측 실행 중:  53%|█████▎    | 53/100 [02:22<02:01,  2.59s/it]